# 상추 생장량 예측 & 환경 제어

전날 재배 환경으로 다음날 생장량을 예측하고, Optuna로 생장을 높이는 환경 조건을 탐색합니다.


In [ ]:
# 데이터 로드 및 모델 학습
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FARMETRY = ROOT
SCRIPTS = FARMETRY / "scripts"
OUT = FARMETRY / "data" / "lettuce_processed"
sys.path.insert(0, str(SCRIPTS))

import importlib
import lettuce_growth_env_analysis as lgea
import lettuce_growth_model as lgm
import lettuce_growth_model_compare as lgc

importlib.reload(lgm)
importlib.reload(lgc)
importlib.reload(lgea)

from lettuce_growth_env_analysis import main as rebuild_data
from lettuce_growth_model_compare import train_best_model_bundle

# 원본 엑셀이 있으면 CSV 재생성, 없으면 저장소에 포함된 CSV 사용
try:
    rebuild_data()
except FileNotFoundError:
    print("원본 엑셀 없음 → data/lettuce_processed 의 기존 CSV 사용")
best = train_best_model_bundle()

merged = best["merged"]
bundle = best["bundle"]
lag_df = best["lag_df"]
summary = best["summary"]
best_model = best["best_model"]

metrics = bundle.metrics
cv = bundle.cv_predictions

print(f"병합 일수: {len(merged)}")
print(f"최고 모델: {best_model}")
print(f"학습 샘플: {metrics['n_samples']}일 (전날 환경 → 다음날 생장)\n")

print("=== 모델 비교 (LOO CV, MAE 순) ===")
print(summary[["model", "MAE_g", "RMSE_g", "R2", "n_samples"]].to_string(index=False))

print("\n=== 예측 정확도 (Leave-One-Out) ===")
print(f"  MAE  : {metrics['MAE_g']:.2f} g/일")
print(f"  RMSE : {metrics['RMSE_g']:.2f} g/일")
print(f"  R²   : {metrics['R2']:.3f}")
print(f"  MAPE : {metrics['MAPE_pct']:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(cv["actual_g"], cv["predicted_g"], alpha=0.8, edgecolors="k", linewidths=0.5)
lim = [cv["actual_g"].min() - 1, cv["actual_g"].max() + 1]
axes[0].plot(lim, lim, "r--", lw=1)
axes[0].set_xlabel("실제 생장량 (g/일)")
axes[0].set_ylabel("예측 생장량 (g/일)")
axes[0].set_title(f"{best_model} — 실제 vs 예측 (R²={metrics['R2']:.2f})")

axes[1].bar(
    range(len(cv)),
    cv["error_g"],
    color=["#e74c3c" if e < 0 else "#2ecc71" for e in cv["error_g"]],
)
axes[1].axhline(0, color="gray", lw=0.8)
axes[1].set_xlabel("샘플 인덱스")
axes[1].set_ylabel("오차 (실제 - 예측, g)")
axes[1].set_title("날짜별 예측 오차")

plt.tight_layout()
plt.show()

merged[["md", "DAT", "biomass_delta_g_mean"]].tail()

- **탐색**: 습도·기온·수온·EC를 10~90% 분위 범위에서 Optuna로 연속 최적화
- **출력**: `current` / `next_control_value` / `bayesian_target` 시나리오별 예상 g/주/일

****에서 기준일·가상 환경을 바꿔 여러 시나리오를 한 번에 비교할 수 있습니다.


In [ ]:
# Optuna 환경 제어
import importlib
import lettuce_growth_env_optuna as lgo

importlib.reload(lgo)
from lettuce_growth_env_optuna import run_env_control_optuna

N_TRIALS = 150  # 늘리면 탐색 정밀도 ↑ (시간 ↑)

optuna_result = run_env_control_optuna(
    bundle=bundle,
    lag_df=lag_df,
    merged=merged,
    n_trials=N_TRIALS,
)

ctrl = optuna_result["control_df"]
imp = optuna_result["improvement_df"]

print(f"\n저장: {OUT / 'env_control_optuna.csv'}")
print(f"저장: {OUT / 'env_control_improvement.csv'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
labels = ["현재", "다음 제어", "장기 목표"]
vals = [
    optuna_result["current_pred_g"],
    optuna_result["next_pred_g"],
    optuna_result["target_pred_g"],
]
colors = ["#9e9e9e", "#ff9800", "#4caf50"]
ax.bar(labels, vals, color=colors)
ax.set_ylabel("예상 생장 (g/주/일)")
ax.set_title("시나리오별 예측 생장")
for i, v in enumerate(vals):
    ax.text(i, v + 0.05, f"{v:.2f}g", ha="center", fontsize=10)

ax = axes[1]
show_ctrl = ctrl.sort_values("target_adjustment", key=abs, ascending=False).head(6)
y_pos = range(len(show_ctrl))
ax.barh(list(y_pos), show_ctrl["target_adjustment"], color="#2196f3")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(show_ctrl["variable"], fontsize=9)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("장기 목표 - 현재")
ax.set_title("환경 변수별 조정량 (목표)")
plt.tight_layout()
plt.show()

ctrl[["variable", "current_value", "next_control_value", "bayesian_target", "advice"]]

아래 `SCENARIOS` 목록을 수정한 뒤 실행하세요.


In [ ]:
# 환경 시나리오 비교
import importlib
import lettuce_growth_env_optuna as lgo

importlib.reload(lgo)
from lettuce_growth_env_optuna import run_env_control_optuna, run_env_scenarios

# 기준일 목록 확인
print("=== 관측일별 환경·실제 생장 ===")
show_cols = ["md", "DAT", "AirHum_pct_min", "AirTemp_C_min", "WaterTemp_C_min", "EC_dS_m_mean", "biomass_delta_g_mean"]
print(merged[show_cols].tail(10).to_string(index=False))

# 비교할 시나리오 (name / base_md / env_override 조합)
SCENARIOS = [
    {"name": "기본_마지막날", "base_md": "03-22"},
    {"name": "생장최고전날", "base_md": "03-10"},
    {"name": "생장급락전날", "base_md": "03-19"},
    {
        "name": "가상_습도낮음",
        "base_md": "03-22",
        "env_override": {"AirHum_pct_min": 30, "WaterTemp_C_min": 19},
    },
    {
        "name": "가상_수온높음",
        "base_md": "03-22",
        "env_override": {"WaterTemp_C_min": 20.5, "EC_dS_m_mean": 2.4},
    },
]

N_TRIALS_SCENARIO = 80

scenario_summary = run_env_scenarios(
    SCENARIOS,
    bundle=bundle,
    lag_df=lag_df,
    merged=merged,
    n_trials=N_TRIALS_SCENARIO,
    verbose_each=False,
)

print("\n=== 시나리오별 Optuna 결과 요약 ===")
print(scenario_summary.to_string(index=False))
print(f"\n저장: {OUT / 'env_scenario_comparison.csv'}")

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(scenario_summary))
w = 0.35
ax.bar([i - w / 2 for i in x], scenario_summary["현재 예측(g/일)"], width=w, label="현재", color="#9e9e9e")
ax.bar([i + w / 2 for i in x], scenario_summary["장기 목표(g/일)"], width=w, label="장기 목표", color="#4caf50")
ax.set_xticks(list(x))
ax.set_xticklabels(scenario_summary["시나리오"], rotation=20, ha="right")
ax.set_ylabel("예상 생장 (g/주/일)")
ax.set_title("환경 시나리오별 예측 생장 비교")
ax.legend()
plt.tight_layout()
plt.show()

detail = run_env_control_optuna(
    bundle=bundle,
    lag_df=lag_df,
    merged=merged,
    base_md="03-22",
    env_override={"AirHum_pct_min": 30, "WaterTemp_C_min": 19},
    scenario_name="가상_습도낮음_상세",
    n_trials=100,
)
detail["control_df"]